# 06 — Mapas de metano CAMS73

Os campos e escalas usam as mesmas funções de `camsaq ch4 analyze`. Meses e anos incompletos são registrados como omitidos; não são preenchidos. Todos os mapas representam uma camada explícita do modelo.

In [ ]:
NOTEBOOK = "06_ch4_maps"
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/config.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Execute dentro do projeto que contém config/config.yaml.")
sys.path.insert(0, str(ROOT / "notebooks"))
from cams_air_quality.config import load_config
from cams_air_quality import ghg_inversion as ghg
from cams_air_quality.extraction import save_table
from available_ch4 import (inventory, open_level, coverage, global_series,
                           point_series, monthly_series, annual_series,
                           reference_cycle, anomaly_series, selected_area)

CONFIG_PATH = Path(os.environ.get("CAMSAQ_CONFIG", ROOT / "config/config.yaml"))
cfg = load_config(CONFIG_PATH)
LEVEL = 1
START = None
END = None
BASELINE_START = None
BASELINE_END = None
BBOX = None
LAT = -23.5505
LON = -46.6333
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = Path(cfg["root"]) / "reports" / "automatic_notebooks" / NOTEBOOK / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
errors = []
print("Saídas:", OUT)

In [ ]:
rows, table = inventory(cfg)
ready = table.loc[table.status == "ready"].to_dict("records") if not table.empty else []
info = coverage(ready)
summary = {"source": "GHG_INVERSION", "level": LEVEL, "status": "no_data",
           "coverage": info, "maps": [], "skipped": []}
if not ready:
    print("Sem CH4 processado. Execute camsaq ch4 download ou import, depois process.")
else:
    try:
        baseline_start, baseline_end = BASELINE_START, BASELINE_END
        if not baseline_start and not baseline_end and len(info["complete_years"]) >= 2:
            baseline_start = f'{info["complete_years"][0]}-01'
            baseline_end = f'{info["complete_years"][-1]}-12'
        summary = ghg.analyze(
            cfg, LEVEL, OUT, start=START, end=END, bbox=BBOX, lat=LAT, lon=LON,
            map_families=["all"], baseline_start=baseline_start,
            baseline_end=baseline_end, scale="shared",
        )
        summary["status"] = "ok"
    except Exception as exc:
        errors.append({"error": str(exc)})
        summary["status"] = "error"
        print("Erro CH4:", exc)
manifest = {"maps": summary.get("maps", []), "skipped": summary.get("skipped", []),
            "coverage": summary.get("coverage", info)}
(OUT / "map_manifest.json").write_text(json.dumps(manifest, indent=2, default=str),
                                       encoding="utf-8")
(OUT / "analysis.json").write_text(json.dumps(summary, indent=2, default=str),
                                   encoding="utf-8")
(OUT / "errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")
for item in manifest["maps"]:
    image = OUT / item["file"]
    if image.is_file():
        display(Image(filename=str(image)))
print("Mapas:", len(manifest["maps"]), "— omitidos:", manifest["skipped"])
print("Erros:", len(errors))

`map_manifest.json` guarda arquivo, família, período e escala; `analysis.json` inclui a cobertura e a referência usada. O desvio padrão é descritivo (`ddof=0`), calculado sobre dias disponíveis.